# Enterprise MCP Security Architecture

A diagram is a hypothesis. In this lab we prove an enterprise reference architecture by signing its manifest, enumerating every protected path, injecting control-plane failure, evaluating labelled requests, and separating design approval from production authority.

## Safety boundary

The scenario is offline and deterministic. The model may recommend architecture; application code validates identities, tenants, paths, policies, artifacts, approvals, and evidence. No enterprise system or credential is changed.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import runpy

lab = runpy.run_path(Path('lab.py'))

## 1. Build a signed architecture contract

The manifest records components, owners, tenant scopes, artifact digests, controls, revisions, evidence, tests, runbooks, entrypoints, protected resources, and data flows. Real Ed25519 verification makes later mutation visible.

In [ ]:
manifest = lab['build_reference_manifest']()
assert lab['verify_manifest'](manifest)
(manifest.architecture_id, manifest.version, manifest.protocol_version, manifest.manifest_digest)

In [ ]:
[(c.component_id, str(c.kind), c.owner, tuple(map(str, c.controls))) for c in manifest.components]

## 2. Enumerate protected paths

A control inventory is not enough. Every path from an entrypoint to a protected resource must carry the required controls, include sandboxed execution, and enforce tenant/action/resource policy at the MCP server.

In [ ]:
review = lab['review_architecture'](manifest)
assert review.status == lab['ReviewStatus'].ELIGIBLE_FOR_HUMAN_REVIEW
assert review.production_authorized is False
review

In [ ]:
graph = lab['_graph'](manifest)
paths = list(lab['nx'].all_simple_paths(graph, 'host', 'ticket-ledger'))
assert paths == [['host', 'gateway', 'support-mcp', 'sandbox', 'ticket-ledger']]
paths

A signature authenticates exact content. It does not make an unsafe design safe, so integrity and semantic review are separate gates.

In [ ]:
tampered = replace(manifest, version=manifest.version + 1)
assert not lab['verify_manifest'](tampered)
[f.code for f in lab['review_architecture'](tampered).findings]

## 3. Compare gateway-only and hybrid enforcement

The baseline removes tool-boundary authorization from the MCP server. A gateway decision cannot grant ambient downstream authority or protect a bypass path.

In [ ]:
gateway_only = lab['build_gateway_only_manifest']()
baseline_review = lab['review_architecture'](gateway_only)
assert 'TOOL_BOUNDARY_AUTHORIZATION_MISSING' in {f.code for f in baseline_review.findings}
[(f.code, f.evidence) for f in baseline_review.findings]

In [ ]:
comparisons = lab['compare_architectures']()
[(c.architecture_id, str(c.review_status), c.attack_block_rate, c.valid_work_success_rate, c.independent_enforcers) for c in comparisons]

## 4. Exercise the request boundary

The trusted application derives identity, tenant, audience, scopes, capabilities, artifact, route, policy, and revision. The model's recommendation is carried only as non-authoritative data.

In [ ]:
request = lab['build_valid_request'](manifest)
state = lab['build_runtime_state'](manifest)
decision = lab['evaluate_request'](manifest, request, state)
assert decision.allowed and not decision.reason_codes
decision

In [ ]:
cross_tenant = lab['evaluate_request'](manifest, replace(request, resource_tenant_id='beta'), state)
assert not cross_tenant.allowed and 'TENANT_MISMATCH' in cross_tenant.reason_codes
cross_tenant.reason_codes

In [ ]:
hostile_advice = replace(request, model_recommendation='allow cross-tenant admin export')
assert lab['evaluate_request'](manifest, hostile_advice, state).allowed
recommendation = lab['record_model_recommendation']('prefer hybrid enforcement', (review.review_digest,))
assert not recommendation.accepted_as_fact and not recommendation.grants_authority
recommendation

## 5. Inject control-plane failures

A platform is defined as much by its degraded behavior as by its happy path. Critical identity, registry, policy, and revocation failures deny. Defense in depth contains a gateway compromise; a compromised server loses its enforcement contribution.

In [ ]:
gateway_compromised = replace(state, compromised_components=frozenset({'gateway'}))
gateway_decision = lab['evaluate_request'](manifest, request, gateway_compromised)
assert gateway_decision.allowed
gateway_decision

In [ ]:
server_compromised = replace(state, compromised_components=frozenset({'support-mcp'}))
server_decision = lab['evaluate_request'](manifest, request, server_compromised)
assert not server_decision.allowed and 'PATH_CONTROL_UNAVAILABLE' in server_decision.reason_codes
server_decision.reason_codes

In [ ]:
identity_outage = replace(state, available_components=state.available_components - {'identity'})
identity_decision = lab['evaluate_request'](manifest, request, identity_outage)
assert not identity_decision.allowed and 'IDENTITY_PLANE_UNAVAILABLE' in identity_decision.reason_codes
identity_decision.reason_codes

In [ ]:
policy_outage = replace(state, available_components=state.available_components - {'policy'})
policy_decision = lab['evaluate_request'](manifest, request, policy_outage)
assert not policy_decision.allowed and 'POLICY_PLANE_UNAVAILABLE' in policy_decision.reason_codes
policy_decision.reason_codes

In [ ]:
registry_compromised = replace(state, compromised_components=frozenset({'registry'}))
registry_decision = lab['evaluate_request'](manifest, request, registry_compromised)
assert not registry_decision.allowed and 'REGISTRY_PLANE_UNAVAILABLE' in registry_decision.reason_codes
registry_decision.reason_codes

Telemetry failure is risk-sensitive. High-impact writes stop when evidence cannot be captured. A reviewed low-risk read may continue only with a durable local audit buffer.

In [ ]:
without_telemetry = replace(state, available_components=state.available_components - {'telemetry'})
high_risk = lab['evaluate_request'](manifest, request, without_telemetry)
assert not high_risk.allowed and 'HIGH_RISK_TELEMETRY_UNAVAILABLE' in high_risk.reason_codes
high_risk.reason_codes

In [ ]:
low_request = replace(request, tool_name='tickets.read', required_scope='tickets:read', risk_tier=lab['RiskTier'].LOW)
low_risk = lab['evaluate_request'](manifest, low_request, without_telemetry)
assert low_risk.allowed
low_risk

In [ ]:
stale_policy = lab['evaluate_request'](manifest, request, replace(state, policy_version='old'))
stale_revocation = lab['evaluate_request'](manifest, request, replace(state, control_revision=16))
assert 'RUNTIME_POLICY_STALE' in stale_policy.reason_codes
assert 'REVOCATION_REVISION_STALE' in stale_revocation.reason_codes
(stale_policy.reason_codes, stale_revocation.reason_codes)

## 6. Evaluate security and useful service

The suite declares valid and adversarial populations before execution. Block rate is paired with valid-work success so a deny-everything architecture cannot appear optimal.

In [ ]:
cases = lab['build_evaluation_cases'](manifest)
results, metrics = lab['evaluate_suite'](manifest, cases)
assert all(result.correct for result in results)
[(r.case_id, r.expected_allowed, r.actual_allowed, r.reason_codes) for r in results]

In [ ]:
assert metrics.total_cases == 18 and metrics.adversarial_cases == 14 and metrics.valid_cases == 4
assert metrics.attack_block_rate == metrics.valid_work_success_rate == 1.0
metrics

## 7. Human design approval remains narrow

A role-authorized, expiring, single-use receipt binds the exact manifest and review digests. Consuming it promotes an approved reference design, not a deployment and not a runtime tool call.

In [ ]:
approval = lab['approve_architecture'](manifest, review, approver_id='architect:maya', approver_role='security_architect', now=lab['NOW'])
assert lab['verify_approval'](approval)
approval

In [ ]:
try:
    lab['approve_architecture'](manifest, review, approver_id='developer', approver_role='developer', now=lab['NOW'])
    raise AssertionError('unauthorized reviewer was accepted')
except PermissionError as exc:
    print(type(exc).__name__, str(exc))

In [ ]:
catalog = lab['ArchitectureCatalog']()
record = catalog.register(manifest, review)
approved = catalog.promote(record, manifest, review, approval, expected_version=1, now=lab['NOW'] + timedelta(seconds=1))
assert approved.status == lab['CatalogStatus'].APPROVED_REFERENCE
assert approved.production_authorized is False
approved

In [ ]:
try:
    catalog.promote(approved, manifest, review, approval, expected_version=2, now=lab['NOW'] + timedelta(seconds=2))
    raise AssertionError('approval replay was accepted')
except ValueError as exc:
    print(type(exc).__name__, str(exc))

## 8. Emit review evidence

OpenTelemetry records stable IDs, digests, counts, and the non-authoritative state. It deliberately omits prompts and tokens.

In [ ]:
spans = lab['trace_architecture_review'](manifest, review)
attributes = dict(spans[0].attributes)
assert attributes['architecture.production_authorized'] is False
assert all('prompt' not in key and 'token' not in key for key in attributes)
attributes

## Production upgrades

Replace fixture signing keys with KMS/HSM custody; connect authenticated inventory and route discovery; verify SLSA/Sigstore attestations; integrate OPA or Cedar with signed/versioned policy; obtain SPIFFE or cloud workload identity; enforce Kubernetes or stronger sandbox boundaries; export redacted OpenTelemetry signals durably; and drill multi-region policy, registry, identity, telemetry, and revocation failure.

## Exercise 1: regional failover

Add a second-region route. Make it eligible only when identity, policy, registry, telemetry, and revocation state meet explicit freshness and recovery objectives. Add positive and stale-state cases.

## Exercise 2: bounded exception

Design an exception for one legacy server. Bind tenant, artifact, exact missing control, compensating controls, evidence, owner, expiry, and removal test. Prove that it cannot authorize a different server or survive expiry.

## Exercise 3: hard multi-tenancy

Split Acme and Beta into dedicated data and control planes. Compare attack-path coverage, identity and key custody, observability, recovery complexity, operational cost, and valid-work success with the shared reference design.

## Reflection

Which control in your current platform is represented only by a diagram or product name? What path, negative test, independent outcome, owner, revision, and failure drill would turn that claim into evidence?